
# **Clustering - Week 2 - Tutorial 1 - Answer Sheet**

You may find the answers within code block and at the end of the demo.

This notebook is intentionally **partly incomplete**.

You will fill in TODOs, change critical parameters, and write short comments directly in the code cells.

The goal is not only to run clustering, but to understand:

- how feature representation changes results
- how nearest-neighbor retrieval depends on distance metrics
- how to choose k for k-Means
- why DBSCAN is sensitive to density parameters
- how to interpret overclustering and underclustering

So, you will compare:

- nearest-neighbor retrieval
- KD-Tree / brute-force search
- k-Means
- DBSCAN
- PCA features
- elbow and silhouette sweeps
- cluster purity matrix
- STEGO-style feature similarity
- CRF-style smoothing intuition

The dataset is the `sklearn` handwritten digits dataset.

Important: true labels are used only for evaluation and interpretation, not for fitting clusters.


# **=== Imports ===**
Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix

np.random.seed(42)


# **=== Helper functions ===**
You do not need to modify these functions.

In [ ]:
def show_digit_grid(images, titles=None, ncols=8, figsize=(10, 4)):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(img.reshape(8, 8), cmap="gray")
        plt.axis("off")
        if titles is not None:
            plt.title(str(titles[i]), fontsize=9)
    plt.tight_layout()
    plt.show()

def plot_2d(X2, labels=None, title="", annotate=False):
    plt.figure(figsize=(7, 5))
    if labels is None:
        plt.scatter(X2[:, 0], X2[:, 1], s=15)
    else:
        unique = np.unique(labels)
        for lab in unique:
            mask = labels == lab
            name = "noise (-1)" if lab == -1 else f"cluster {lab}"
            plt.scatter(X2[mask, 0], X2[mask, 1], s=15, label=name)

        if annotate:
            for lab in unique:
                mask = labels == lab
                if np.sum(mask) == 0:
                    continue
                text = "noise" if lab == -1 else str(lab)
                plt.text(
                    X2[mask, 0].mean(),
                    X2[mask, 1].mean(),
                    text,
                    fontsize=12,
                    fontweight="bold",
                    ha="center",
                    va="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.75)
                )

        if len(unique) <= 15:
            plt.legend(title="Arbitrary cluster IDs", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.title(title)
    plt.xlabel("PCA component 1")
    plt.ylabel("PCA component 2")
    plt.tight_layout()
    plt.show()

def cluster_composition(labels, y_true, max_clusters=15):
    # This function uses true labels only AFTER clustering, for interpretation.
    for c in np.unique(labels)[:max_clusters]:
        idx = np.where(labels == c)[0]
        digits, counts = np.unique(y_true[idx], return_counts=True)
        order = np.argsort(counts)[::-1]
        summary = ", ".join([f"{digits[i]}:{counts[i]}" for i in order[:5]])
        name = "noise (-1)" if c == -1 else f"cluster {c}"
        print(f"{name:>12} | size={len(idx):>4} | true-digit composition: {summary}")

def plot_purity_matrix(labels, y_true, title="Cluster purity matrix"):
    # Rows: predicted clusters
    # Columns: true digit labels
    # Colors: fraction of each true digit inside a cluster
    valid = labels != -1
    cm = confusion_matrix(labels[valid], y_true[valid])
    row_sums = cm.sum(axis=1, keepdims=True)
    purity = cm / np.maximum(row_sums, 1)

    plt.figure(figsize=(8, 5))
    plt.imshow(purity, aspect="auto")
    plt.colorbar(label="fraction within cluster")
    plt.xlabel("true digit label")
    plt.ylabel("predicted cluster ID")
    plt.title(title)
    plt.xticks(range(10))
    plt.tight_layout()
    plt.show()

    return purity


# **=== 1. Load dataset ===**
- Dataset: handwritten digits from sklearn.
- Each image is 8 x 8 pixels, flattened into 64 numbers.
- IMPORTANT: y_true is used only for interpretation, not for fitting clusters.


In [ ]:
digits = load_digits()
X_raw = digits.data.astype(np.float32)
y_true = digits.target

print("Data shape:", X_raw.shape)
print("Label shape:", y_true.shape)

# TODO 1:
# Inspect the sample images below.
# Write one short comment here:
# COMMENT: What visual similarities/differences do you notice between digits?
# ANSWER:
# The digits are low resolution and some classes look visually similar.
# Digits such as 1/7, 3/8, and 4/9 may overlap in feature space.
# This means clustering may group samples by writing style or shape instead of
# exact semantic labels.

sample_idx = np.random.choice(len(X_raw), 24, replace=False)
show_digit_grid(X_raw[sample_idx], titles=y_true[sample_idx], ncols=8)


# **=== 2. PCA feature representation ===**
- PCA reduces 64 pixel dimensions into fewer dimensions.
- You must choose how many PCA components to keep.

In [ ]:
# TODO 2A:
# Try N_COMPONENTS = 5, 10, 20, 40.
# Which one seems like a good balance between compression and information?
# ANSWER:
# Typical observations:
# - 5 PCA components:
#   - strong compression
#   - information loss
#   - weaker clustering
# - 10 components:
#   - usable but still compressed
# - 20 components:
#   - good balance between compression and information
# - 40 components:
#   - preserves more detail but also more noise
# N_COMPONENTS = 20

X_scaled = StandardScaler().fit_transform(X_raw)

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# 2D features only for visualization
X_vis = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

explained = pca.explained_variance_ratio_.sum()
print("PCA feature shape:", X_pca.shape)
print("Explained variance kept:", round(explained, 3))

plt.figure(figsize=(6, 4))
pca40 = PCA(n_components=40, random_state=42).fit(X_scaled)
plt.plot(np.cumsum(pca40.explained_variance_ratio_), marker="o")
plt.xlabel("number of PCA components")
plt.ylabel("cumulative explained variance")
plt.title("PCA explained variance")
plt.grid(True)
plt.show()

plot_2d(X_vis, y_true, "2D PCA visualization, colored by true digit label")

# TODO 2B:
# Write your answer:
# - Chosen N_COMPONENTS:
# - Why did you choose it?
# - What might happen if N_COMPONENTS is too small?
# - What might happen if N_COMPONENTS is too large?
# ANSWER:
# Chosen PCA dimension:
# - Around 20 components is usually a good balance.
# Why:
# - Keeps most useful variance while reducing dimensionality and noise.
# If PCA dimension is too small:
# - Important digit structure may disappear.
# If PCA dimension is too large:
# - More noise may remain and clustering can become less stable.


# **=== 3. Nearest-neighbor retrieval ===**
- This is similarity search.
- The model retrieves images that are close to a query according to a distance metric.

In [ ]:
# TODO 3A:
# Change these three settings and observe the retrieved images.
# Try:
#   X_feature = X_scaled or X_pca
#   metric = "euclidean", "cosine", or "manhattan"
#   algorithm = "brute", "kd_tree", or "ball_tree"
#
# Note: kd_tree does not support all metrics. If it fails, try brute.
# ANSWER:
# Expected observations:
# - Euclidean distance on PCA features usually retrieves visually similar digits.
# - Cosine distance may also work well.
# - KD-Tree is not always useful in higher dimensions.
# - Some metrics are incompatible with KD-Tree.

X_feature = X_pca
metric = "euclidean"
algorithm = "brute"

query_index = 0
n_neighbors = 8

nn = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric=metric,
    algorithm=algorithm
)

nn.fit(X_feature)
distances, indices = nn.kneighbors(X_feature[query_index:query_index+1])

print("Query true label, for interpretation only:", y_true[query_index])
print("Retrieved labels, for interpretation only:", y_true[indices[0]])
print("Distances:", np.round(distances[0], 3))

show_digit_grid(
    X_raw[indices[0]],
    titles=[f"y={y_true[i]}" for i in indices[0]],
    ncols=n_neighbors,
    figsize=(12, 2)
)

# TODO 3B:
# Write your answer:
# - Which feature space did you try?
# - Which metric looked best?
# - Did KD-Tree/BallTree work for your metric?
# - Why are these results not the same as classification?
# ANSWER:
# Example answer:
# - Feature space used: PCA features
# - Best metric: Euclidean or cosine
# - KD-Tree worked only with some metrics
# - This is retrieval, not classification, because the method only returns
# nearby samples and does not learn class boundaries.


# **=== 4. k-Means: elbow + silhouette sweep ===**
- We do not know the best k in advance.
- Here you will compare several possible k values.

In [ ]:
ks = range(2, 21)

# TODO 4A:
# Complete the loop below.
# For each k:
#   1. create KMeans
#   2. fit_predict on X_pca
#   3. store model.inertia_
#   4. store silhouette_score(X_pca, labels)
# ANSWER:
# The loop should:
# 1. Create k-Means
# 2. Fit and predict labels
# 3. Store inertia
# 4. Store silhouette score
# Expected:
# - Inertia always decreases as k increases.
# - Silhouette may peak at a different k than expected semantically.

inertias = []
silhouettes = []

for k in ks:
    # TODO: create the k-Means model
    model = KMeans(n_clusters=k, random_state=42, n_init=10)

    # TODO: fit and predict labels
    labels = model.fit_predict(X_pca)

    # TODO: save inertia
    inertias.append(model.inertia_)

    # TODO: save silhouette score
    silhouettes.append(silhouette_score(X_pca, labels))

# TODO 4B:
# Plot inertia curve and silhouette curve.
# Question: Do both curves suggest the same k?
# ANSWER:
# Expected:
# - Elbow curve shows where inertia improvement slows down.
# - Silhouette curve shows cluster separation quality.
# Important:
# - Elbow and silhouette may not suggest the same k.

plt.figure(figsize=(6, 4))
plt.plot(list(ks), inertias, marker="o")
plt.xlabel("k")
plt.ylabel("inertia")
plt.title("Elbow sweep for k-Means")
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(list(ks), silhouettes, marker="o")
plt.xlabel("k")
plt.ylabel("silhouette score")
plt.title("Silhouette sweep for k-Means")
plt.grid(True)
plt.show()

best_k_by_silhouette = list(ks)[int(np.argmax(silhouettes))]
print("Best k by silhouette:", best_k_by_silhouette)

# TODO 4C:
# Write your answer:
# - What k does the elbow suggest?
# - What k does silhouette suggest?
# - Are they the same?
# - Why is the best mathematical k not necessarily the best semantic k?
# ANSWER:
# Example interpretation:
# - Elbow may suggest a moderate k value.
# - Silhouette may prefer another k.
# - Not necessarily.
# - Best geometric clustering is not always the best semantic clustering because
# these metrics do not directly know true labels.


# **=== 5. k-Means clustering and interpretation ===**

In [ ]:
# TODO 5A:
# Choose K based on your reasoning above.
# Try at least two values, e.g. K = 10 and K = 20.
# ANSWER:
# Expected behavior:
# - Small k:
#   - underclustering
#   - multiple digits merged
# - Large k:
#   - overclustering
#   - one digit split into several style clusters

Typical mixed digits:
- 3 and 8
- 4 and 9
- 1 and 7
K = 10

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_pca)

print("Silhouette:", round(silhouette_score(X_pca, labels_kmeans), 3))
print("ARI vs true labels, for interpretation only:", round(adjusted_rand_score(y_true, labels_kmeans), 3))
print()
print("Cluster composition:")
cluster_composition(labels_kmeans, y_true)

plot_2d(X_vis, labels_kmeans, f"k-Means clustering, k={K}", annotate=True)
plot_purity_matrix(labels_kmeans, y_true, title=f"k-Means purity matrix, k={K}")

# TODO 5B:
# Write your answer:
# - Did k-Means undercluster? Example:
# - Did k-Means overcluster? Example:
# - Which digits were mixed together?
# - Why do cluster IDs/colors not correspond to digit labels?
# ANSWER:
# Example:
# - Underclustering occurred with small k because several digits were merged
# together.
# - Overclustering occurred with large k because one digit was split into
# multiple clusters.
# - 1 and 7, 3 and 8, 4 and 9, 5 and 6, sometimes 0 and 6. These digits are
# mixed because they look visually similar in the feature space.
# - Cluster IDs/colors are arbitrary and do not correspond to digit labels.


# **=== 6. DBSCAN parameter tuning ===**
- DBSCAN discovers density-based clusters.
- It can also mark points as noise with label -1.

In [ ]:
# TODO 6A:
# Try several combinations:
#   eps = 1.5, 2.0, 2.5, 3.0, 4.0
#   min_samples = 3, 5, 10
#
# Goal:
# - avoid all points becoming noise
# - avoid everything merging into one cluster
# - inspect whether clusters look meaningful
# ANSWER:
# Expected DBSCAN behavior:
# - Small eps:
#   - many points become noise
# - Large eps:
#   - clusters merge together
# - Larger min_samples:
#   - stricter density requirement
#   - more noise points

eps = 3.0
min_samples = 5

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels_dbscan = dbscan.fit_predict(X_pca)

unique = np.unique(labels_dbscan)
n_noise = np.sum(labels_dbscan == -1)
n_clusters = len(unique) - (1 if -1 in unique else 0)

print("Unique labels:", unique)
print("Number of clusters:", n_clusters)
print("Noise count:", n_noise)
print("ARI vs true labels, for interpretation only:", round(adjusted_rand_score(y_true, labels_dbscan), 3))

if n_clusters >= 2:
    print("Silhouette:", round(silhouette_score(X_pca, labels_dbscan), 3))
else:
    print("Silhouette unavailable because DBSCAN found fewer than 2 clusters.")

print()
print("Cluster composition:")
cluster_composition(labels_dbscan, y_true)

plot_2d(X_vis, labels_dbscan, f"DBSCAN, eps={eps}, min_samples={min_samples}", annotate=True)
plot_purity_matrix(labels_dbscan, y_true, title=f"DBSCAN purity matrix, eps={eps}")

# TODO 6B:
# Write your answer:
# - What happened when eps was too small?
# - What happened when eps was too large?
# - What happened when min_samples increased?
# - Did DBSCAN work better or worse than k-Means on this dataset?
# - Why might DBSCAN struggle on digit PCA features?
# ANSWER:
# Example interpretation:
# - DBSCAN was sensitive to eps.
# - Too small eps produced many noise points.
# - Too large eps merged clusters.
# - DBSCAN often worked worse than k-Means on this dataset because digit PCA
# features are not strongly density-separated.
# DBSCAN may struggle because digit PCA features are not clearly density-separated.
# Some digits overlap visually, and different digit classes may have different
# densities. Therefore, small eps can create many noise points, while large eps
# can merge different digits.


# **=== 7. Feature representation comparison ===**
- Deep clustering depends heavily on feature quality.
- Here we compare k-Means on several feature representations.

In [ ]:
# TODO 7A:
# Add or remove feature spaces if you want.
# ANSWER:
# Expected feature comparison:
# - Raw pixels:
#   - weaker clustering
# - PCA features:
#   - cleaner structure
# - Normalized PCA:
#   - useful for cosine similarity

feature_spaces = {
    "raw pixels": X_raw,
    "scaled pixels": X_scaled,
    "PCA features": X_pca,
    "normalized PCA": normalize(X_pca),
}

for name, feat in feature_spaces.items():
    labels = KMeans(n_clusters=10, random_state=42, n_init=10).fit_predict(feat)

    n_clusters = len(np.unique(labels))
    sil = silhouette_score(feat, labels)
    ari = adjusted_rand_score(y_true, labels)

    print(f"{name:>15} | clusters={n_clusters:>2} | silhouette={sil:.3f} | ARI={ari:.3f}")

# TODO 7B:
# Write your answer:
# - Which feature representation worked best according to silhouette?
# - Which worked best according to ARI?
# - Did both metrics agree?
# - What does this teach us about deep clustering?
# ANSWER:
# Example interpretation:
# - PCA or normalized PCA usually performed best.
# - Silhouette and ARI may disagree because one measures geometry and the other
# measures agreement with labels.
# - This demonstrates why feature quality is critical in deep clustering.


# **=== 8. STEGO-style feature similarity ===**
- STEGO uses self-supervised features and feature correspondences.
- Here we only visualize feature similarity between samples.

In [ ]:
# TODO 8A:
# Try changing subset_size to 40 or 100.
# ANSWER:
# Expected similarity matrix:
# - Semantically related samples should form block-like structures with high
# similarity.
subset_size = 80

subset = np.random.choice(len(X_pca), size=subset_size, replace=False)
F = normalize(X_pca[subset])
similarity = F @ F.T

plt.figure(figsize=(6, 5))
plt.imshow(similarity, aspect="auto")
plt.colorbar(label="cosine similarity")
plt.title("Feature similarity matrix")
plt.xlabel("sample index")
plt.ylabel("sample index")
plt.show()

print("True labels for interpretation only:")
print(y_true[subset])

# TODO 8B:
# Write your answer:
# - What would a good similarity matrix look like?
# - Why is feature similarity important for STEGO?
# - Why is STEGO not just clustering raw pixels?
# ANSWER:
# Example interpretation:
# - STEGO relies on self-supervised feature similarity and correspondence.
# - It is not simply clustering raw pixels.
# - Better features create more meaningful semantic grouping.


# **=== 9. CRF-style smoothing intuition ===**
- A CRF can refine segmentation by encouraging local consistency.
- Here we use a very simple 1D smoothing example.

In [ ]:
# TODO 9A:
# Change this noisy label sequence.
# Try adding isolated wrong labels and see whether smoothing fixes them.
# ANSWER:
# Expected:
# - Smoothing fixes isolated noisy labels.
# - Strong smoothing may damage boundaries or thin structures.

labels = np.array([0,0,0,1,0,0,1,1,1,1,0,1,1,2,2,2,1,2,2,2])

def majority_smooth_1d(labels, window=3):
    pad = window // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    new = []
    for i in range(len(labels)):
        local = padded[i:i+window]
        values, counts = np.unique(local, return_counts=True)
        new.append(values[np.argmax(counts)])
    return np.array(new)

smoothed = majority_smooth_1d(labels, window=3)

print("Original noisy labels:")
print(labels)
print("Smoothed labels:")
print(smoothed)

plt.figure(figsize=(10, 2))
plt.plot(labels, marker="o", label="noisy")
plt.plot(smoothed, marker="x", label="smoothed")
plt.title("CRF-style intuition: local consistency")
plt.legend()
plt.show()

# TODO 9B:
# Write your answer:
# - What kind of error did smoothing fix?
# - What kind of structure could smoothing damage?
# - Why can CRF-like refinement help segmentation?
# ANSWER:
# Example interpretation:
# - CRF-like smoothing improves local consistency.
# - Too much smoothing can oversmooth semantic boundaries or remove small
# details.
# - CRF-like refinement helps segmentation by enforcing local consistency:
# nearby and visually similar pixels are encouraged to have similar labels.
# This can remove isolated noisy predictions and make segment boundaries
# cleaner, but too much smoothing may erase small objects or sharp boundaries.




# Final mini-report

Before leaving, fill these answers in your notebook:

1. Best nearest-neighbor setting:
2. Chosen PCA dimension and why:
3. Chosen k for k-Means and why:
4. Evidence of overclustering or underclustering:
5. Best DBSCAN parameters:
6. Why DBSCAN succeeded or failed:
7. Best feature representation:
8. Connection to deep clustering / STEGO:
9. One thing that surprised you:


# Example Final Mini-Report -- Answers

1. Best nearest-neighbor setting: Euclidean/cosine on PCA features.
2. Chosen PCA dimension: Around 20.
3. Chosen k for k-Means: Around 10.
4. Overclustering/underclustering: Small k merged digits, large k split styles.
5. DBSCAN: Sensitive to eps.
6. Best feature representation: PCA or normalized PCA.
7. Deep clustering insight: Better features improve clustering.
8. STEGO insight: Semantic similarity comes from learned features.


# Demo Answer Sheet / Instructor Notes

This notebook contains suggested answers and interpretations for the clustering tutorial. Numerical results may vary slightly depending on initialization and parameter choices.

## PCA

PCA reduces the original 64-dimensional digit vectors into fewer informative dimensions.

Typical observations:
- 5 components: too compressed, information loss
- 10 components: usable but limited
- 20 components: good balance
- 40 components: preserves more information but also more noise

Good answer:
> Around 20 PCA components often provide a good trade-off between compression and retained information.

## Nearest Neighbour Retrieval

Expected observations:
- Euclidean or cosine distance on PCA features usually retrieves visually similar digits.
- KD-Tree is not always beneficial in higher-dimensional spaces.
- Retrieval is not classification; the method simply finds nearby samples.

Good answer:
> Euclidean or cosine distance on PCA features gave the most visually meaningful neighbors.

## Elbow + Silhouette Sweep

## Elbow:
- inertia always decreases as k increases
- look for the point where improvement slows down

## k-Means Inertia (Elbow Method)

k-Means minimizes the within-cluster squared distance:

$$
inertia(k) = \sum_{i=1}^{k}\sum_{x \in C_i} \|x - \mu_{i}\|^2
$$

where:

- $x$ = data point in cluster $i$
- $\mu_{i}$ = centroid of the assigned cluster $i$
- ${C_i}$ = cluster $i$
- $k$ = number of clusters

Lower inertia means points are closer to their cluster centers.

--------------------

## Silhouette:
- measures geometric separation
- higher values are usually better geometrically

## Silhouette Score

$$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$$

Higher silhouette means:
- points are close to their own cluster
- and far from other clusters

Important:
The mathematically best k is not always the semantically best k.

Good answer:
> Elbow and silhouette may suggest different k values because they measure different properties.

## k-Means

Expected behavior:
- small k: underclustering
- large k: overclustering

Important:
Cluster IDs and colors are arbitrary.

Typical mixed digits:
- 3 and 8
- 4 and 9
- 1 and 7

Good answer:
> Increasing k can split one semantic digit into multiple style-based clusters.

## Cluster Purity Matrix

Rows = predicted clusters
Columns = true digit labels

Interpretation:
- one strong cell in a row: pure cluster
- multiple strong cells: mixed cluster
- same digit across many rows: overclustering

## ARI (Adjusted Rand Index)

ARI compares predicted clusters with true labels.

Interpretation:
- 1.0 = perfect agreement
- around 0 = random clustering
- negative = worse than random

Important:
ARI is only used after clustering for evaluation.

## DBSCAN

Expected behavior:
- eps too small -> many noise points
- eps too large -> clusters merge
- larger min_samples -> stricter density requirement

DBSCAN may struggle because digit PCA features are not strongly density-separated.

Good answer:
> DBSCAN was sensitive to eps and often produced unstable results on this dataset.

## Feature Representation Comparison

Expected observations:
- raw pixels: weaker representation
- PCA features: often cleaner clusters
- normalized PCA: useful for cosine similarity

Important:
Feature quality often matters more than the clustering algorithm.

## STEGO-style Feature Similarity

A good similarity matrix should show stronger similarity among semantically related samples.

STEGO uses:
- self-supervised features
- feature correspondences
- semantic consistency

Important:
STEGO is not just clustering raw pixels.

## CRF-style Smoothing

CRF-like smoothing:
- fixes isolated noisy predictions
- can oversmooth thin structures and boundaries

Good answer:
> CRF refinement improves local consistency but must preserve semantic boundaries.